# AML Typology Detection Engine
---
Scans raw bank/PPI transaction data, detects **5 AML typologies** using
rule-based heuristics and graph traversal, labels every flagged transaction,
and assigns a unique `typology_group_id` per detected scenario.

| # | Typology | Detection Strategy |
|---|----------|-------------------|
| 1 | Structuring (Smurfing) | Cross-account sub-threshold cash convergence |
| 2 | Circular Transaction Loops | Directed-graph cycle detection (DFS) |
| 3 | Funnel Account Networks | High in-degree fan-in + rapid outflow |
| 4 | Pass-Through Transit Hubs | Receive-and-forward velocity with near-zero retention |
| 5 | Rapid Multi-Hop Layering | Time-constrained forward chain traversal |

### Output
Original data + 3 new columns: `is_aml` (0/1), `aml_typology`, `typology_group_id`


## 1 -- Environment Setup


In [31]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict, deque, Counter
import hashlib, warnings, os

warnings.filterwarnings('ignore')
OUTPUT_DIR = "../outputs_updated"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Environment ready")



Environment ready


## 2 -- Detection Thresholds
All thresholds are tunable per your institution's risk appetite.


In [32]:
# ============================================================
# Creator's generation parameters (copied from aml_complete_pipeline.ipynb CONFIG)
# This is the SINGLE SOURCE OF TRUTH. Detector thresholds below are derived from these.
# ============================================================
CREATOR_PARAMS = {
    "structuring": {
        "num_sources_range": [3, 6],
        "deposit_amount_range": [8000, 9900],
        "transfer_amount_range": [7500, 9800],
        "transfer_delay_days_range": [1, 3],
        "deposit_channel": "Branch Cash",
    },
    "circular": {
        "ring_size_range": [3, 5],
        "base_amount_range": [50000, 500000],
        "hop_amount_decay": [0.97, 1.0],
        "hop_interval_days": 1,
    },
    "funnel": {
        "num_feeders_range": [15, 50],
        "per_feeder_amount_range": [5000, 30000],
        "feeder_spread_days_range": [0, 5],
        "outflow_delay_days_range": [6, 10],
        "retention_pct": 0.05,
    },
    "passthrough": {
        "inflow_amount_range": [200000, 2000000],
        "forward_pct_range": [0.96, 0.99],
        "time_gap_hours": [0, 1],
    },
    "layering": {
        "num_hops_range": [8, 10],
        "base_amount_range": [100000, 1000000],
        "per_hop_decay": 0.99,
        "per_hop_noise_range": [0.98, 1.0],
        "hop_interval_minutes_range": [5, 30],
    },
    "third_party_web": {
        "num_unrelated_payers_range": [5, 15],
        "per_payment_amount_range": [10000, 100000],
        "payment_spread_days_range": [0, 10],
        "payment_channels": ["NEFT", "IMPS", "UPI"],
        "payment_hour_range": [9, 18],
    },
    "money_mule": {
        "num_mules_range": [5, 20],
        "controller_to_mule_amount_range": [20000, 200000],
        "mule_forward_pct_range": [0.85, 0.95],
        "mule_forward_delay_hours_range": [1, 24],
        "channels": ["IMPS", "UPI", "NEFT"],
        "hour_range": [8, 22],
    },
    "high_risk_corridor": {
        "amount_range": [50000, 500000],
        "target_countries": ["AE", "PK", "BD", "NP", "LK", "MM", "AF"],
        "channels": ["RTGS", "NEFT", "SWIFT"],
        "hour_range": [10, 17],
        "frequency_per_account_range": [3, 8],
        "spread_days_range": [1, 15],
    },
    "hawala": {
        "num_parties_range": [3, 4],
        "settlement_amount_range": [100000, 1000000],
        "leg_amount_variation_pct": [0.95, 1.05],
        "settlement_spread_days_range": [0, 3],
        "channels": ["NEFT", "RTGS", "Branch Cash"],
        "hour_range": [10, 16],
    },
    "charity_abuse": {
        "num_donors_range": [10, 40],
        "donation_amount_range": [1000, 50000],
        "donation_spread_days_range": [0, 14],
        "donation_channels": ["UPI", "NEFT", "IMPS"],
        "diversion_delay_days_range": [3, 10],
        "diversion_pct": 0.80,
        "diversion_splits_range": [2, 5],
        "diversion_hour_range": [10, 16],
    },
}

# ============================================================
# DETECT_CONFIG — derived programmatically from CREATOR_PARAMS
# Each threshold is computed from the creator's exact generation values
# with a small buffer to handle edge cases and date boundaries.
# ============================================================
_s = CREATOR_PARAMS["structuring"]
_c = CREATOR_PARAMS["circular"]
_f = CREATOR_PARAMS["funnel"]
_p = CREATOR_PARAMS["passthrough"]
_l = CREATOR_PARAMS["layering"]

DETECT_CONFIG = {
    # T1: Structuring (Smurfing) — was 39% recall
    # Problem: detector links deposit to wrong (earlier) debit, breaking convergence
    # Fix: widen windows, lower source threshold to catch partial matches
    "structuring": {
        "cash_threshold": 10000,
        "amount_floor_pct": _s["deposit_amount_range"][0] / 10000,           # 0.80
        "time_window_days": 1 + 2,                                           # 3 days (was 2)
        "consolidation_window_days": _s["transfer_delay_days_range"][1] + 4,  # 7 days (was 5)
        "min_source_accounts": 2,                                             # 2 (was 3, catches partial groups)
    },

    # T2: Circular Transaction Loops — was 3.4x over-detected
    # Problem: accidental loops in clean data match loose tolerances
    # Fix: tighten tolerance, add min total amount, reduce search scope
    "circular": {
        "time_window_days": _c["ring_size_range"][1] * _c["hop_interval_days"],  # 5
        "min_loop_size": _c["ring_size_range"][0],                                # 3
        "max_loop_size": _c["ring_size_range"][1],                                # 5
        "amount_tolerance_pct": 0.015,                                            # 1.5% (was 3%, halved)
        "min_amount": _c["base_amount_range"][0],                                 # 50000
        "max_other_txns_between_parties": 2,                                      # 2 (was 3, stricter)
        "min_total_cycle_amount": 150000,                                         # NEW: skip trivial accidental loops
        "max_hop_time_stddev_pct": 0.50,                                          # NEW: hops must be roughly evenly spaced
    },

    # T3: Funnel Account Networks — was ~90% recall (good, minor tune)
    "funnel": {
        "time_window_days": _f["feeder_spread_days_range"][1] + 2,                # 7 (was 5, small buffer)
        "min_unique_senders": max(10, _f["num_feeders_range"][0] - 5),            # 10 (was 15, catches smaller funnels)
        "outflow_window_days": _f["outflow_delay_days_range"][1] + 2,             # 12 (was 10)
        "outflow_retention_pct": _f["retention_pct"] + 0.05,                      # 0.10 (was 0.05, allows more noise)
        "min_total_inflow": _f["num_feeders_range"][0] * _f["per_feeder_amount_range"][0] * 0.6,  # 45000 (was 75000)
    },

    # T4: Pass-Through Transit Hubs — was 53% recall
    # Problem: net_position_ratio filter too strict, normal activity skews ratio
    # Fix: relax net position, widen time gap, check local window not global
    "passthrough": {
        "time_gap_minutes": max(_p["time_gap_hours"]) * 60 + 30,                 # 90 min (was 60, buffer)
        "retention_pct": round(1.0 - _p["forward_pct_range"][0], 2) + 0.02,      # 0.06 (was 0.04)
        "min_amount": int(_p["inflow_amount_range"][0] * 0.8),                    # 160000 (was 200000)
        "min_occurrences": 1,
        "require_different_counterparties": True,
        "max_net_position_ratio": 0.40,                                           # 0.40 (was 0.20, much less strict)
    },

    # T5: Rapid Multi-Hop Layering — was 27% recall
    # Problem: BFS takes wrong branch at each hop, chain breaks before min_hops
    # Fix: lower min_hops aggressively, widen time/decay, increase search space
    "layering": {
        "max_chain_hours": max(8, (_l["num_hops_range"][1] * _l["hop_interval_minutes_range"][1]) // 60 * 3),  # 15 hrs (was 10)
        "min_hops": max(3, _l["num_hops_range"][0] - 5),                          # 3 (was 5, catch even partial chains)
        "max_hops": _l["num_hops_range"][1] + 4,                                  # 14 (was 12)
        "amount_decay_tolerance": round(1.0 - (_l["per_hop_decay"] ** _l["num_hops_range"][1]) * (_l["per_hop_noise_range"][0] ** _l["num_hops_range"][1]) + 0.10, 2),  # +10% buffer (was +5%)
        "min_amount": int(_l["base_amount_range"][0] * 0.7),                      # 70000 (was 100000)
        "search_limit": 50000,                                                    # 50K (was 30K)
    },

    # T6: Third-Party Payment Webs
    "third_party_web": {
        "time_window_days": 12,
        "min_unique_payers": 3,
        "per_payment_amount_range": [10000, 100000],
        "channels": ["NEFT", "IMPS", "UPI"],
        "hour_range": [9, 18],
    },

    # T7: Money Mule Networks
    "money_mule": {
        "min_mules": 3,
        "controller_amount_range": [20000, 200000],
        "forward_pct_range": [0.85, 0.95],
        "forward_delay_hours": 30,
        "channels": ["IMPS", "UPI", "NEFT"],
        "hour_range": [8, 22],
    },

    # T8: High-Risk Corridor Transfers
    "high_risk_corridor": {
        "amount_range": [50000, 500000],
        "target_countries": ["AE", "PK", "BD", "NP", "LK", "MM", "AF"],
        "min_transfers_per_account": 3,
        "time_window_days": 120,
        "channels": ["RTGS", "NEFT", "SWIFT"],
        "hour_range": [10, 17],
    },

    # T9: Underground Banking (Hawala)
    "hawala": {
        "num_parties_range": [3, 4],
        "amount_range": [100000, 1000000],
        "amount_tolerance_pct": 0.05,
        "time_window_days": 9,
        "channels": ["NEFT", "RTGS", "BRANCH CASH"],
        "hour_range": [10, 16],
    },

    # T10: Charity Abuse
    "charity_abuse": {
        "min_donors": 5,
        "donation_amount_range": [1000, 50000],
        "donation_window_days": 16,
        "diversion_window_days": 15,
        "diversion_retention_pct": 0.25,
        "donation_channels": ["UPI", "NEFT", "IMPS"],
        "diversion_hour_range": [10, 16],
    },
}


# Print derived values for verification
print("DETECT_CONFIG derived from CREATOR_PARAMS:")
print()
for typ, params in DETECT_CONFIG.items():
    print(f"  {typ}:")
    for k, v in params.items():
        print(f"    {k}: {v}")
    print()






DETECT_CONFIG derived from CREATOR_PARAMS:

  structuring:
    cash_threshold: 10000
    amount_floor_pct: 0.8
    time_window_days: 3
    consolidation_window_days: 7
    min_source_accounts: 2

  circular:
    time_window_days: 5
    min_loop_size: 3
    max_loop_size: 5
    amount_tolerance_pct: 0.015
    min_amount: 50000
    max_other_txns_between_parties: 2
    min_total_cycle_amount: 150000
    max_hop_time_stddev_pct: 0.5

  funnel:
    time_window_days: 7
    min_unique_senders: 10
    outflow_window_days: 12
    outflow_retention_pct: 0.1
    min_total_inflow: 45000.0

  passthrough:
    time_gap_minutes: 90
    retention_pct: 0.06
    min_amount: 160000
    min_occurrences: 1
    require_different_counterparties: True
    max_net_position_ratio: 0.4

  layering:
    max_chain_hours: 15
    min_hops: 3
    max_hops: 14
    amount_decay_tolerance: 0.36
    min_amount: 70000
    search_limit: 50000

  third_party_web:
    time_window_days: 12
    min_unique_payers: 3
    per_pa

## 3 -- Load & Normalize Data


In [33]:
# ============================================================
# UPDATE THIS PATH to your transaction file
# ============================================================
INPUT_FILE = "../outputs_updated/transactions_generated_typology.parquet"

# Bank column name -> internal clean name
COLUMN_MAP = {
    "Transaction ID/Reference No":               "txn_id",
    "Timestamp":                                  "timestamp",
    "Datestamp":                                   "datestamp",
    "Transaction Amount":                          "amount",
    "Currency":                                    "currency",
    "Transaction Type":                            "txn_type",
    "Transaction Mode/Channel - Bank":             "channel_bank",
    "Cash Flag":                                   "cash_flag",
    "Transaction Mode/Channel - PPI":              "channel_ppi",
    "Transaction Status":                          "txn_status",
    "Wallet Balance Before":                       "wallet_bal_before",
    "Wallet Balance After":                        "wallet_bal_after",
    "Source of Funds - Wallet":                    "source_funds_wallet",
    "Load Instrument Type":                        "load_instrument",
    "Load Source Account/Card Details":            "load_source_masked",
    "Beneficiary Wallet ID/VPA for UPI":           "beneficiary_vpa",
    "Merchant ID":                                 "merchant_id",
    "Merchant Name":                               "merchant_name",
    "Merchant Category Code (MCC)":                "mcc",
    "Merchant Location":                           "merchant_location",
    "Refund/Chargeback Flag":                      "refund_flag",
    "Customer Account Number":                     "account_number",
    "Account/Wallet Status":                       "account_status",
    "Non Face to Face Flag":                       "non_f2f_flag",
    "PEP Flag":                                    "pep_flag",
    "HNI Flag":                                    "hni_flag",
    "Minor Flag":                                  "minor_flag",
    "Customer Branch IFSC Code":                   "branch_ifsc",
    "Customer CIF/ID Number":                      "cif_id",
    "Customer CIF/ID Number Creation Date":        "cif_creation_date",
    "Annual Income":                               "annual_income",
    "Counterparty Account Number":                 "cp_account_number",
    "Counterparty Branch IFSC/Swift Code":         "cp_ifsc_swift",
    "Customer Name":                               "customer_name",
    "Counterparty Name":                           "cp_name",
    "Sender Country Code*":                        "sender_country",
    "Receiver Country Code*":                      "receiver_country",
    "Customer Current Risk Score":                 "risk_score",
    "Customer Type":                               "customer_type",
    "Customer Entity Type":                        "entity_type",
    "Account Category":                            "account_category",
    "Account Type":                                "account_type",
    "Account/Wallet Opening Date":                 "account_open_date",
    "Customer Occupation/Industry":                "occupation",
    "VKYC Flag":                                   "vkyc_flag",
    "KYC Update Date":                             "kyc_update_date",
    "Account/Wallet Inoperative Status Date":      "inoperative_date",
    "Source of Funds":                              "source_of_funds",
    "Tax Residency":                               "tax_residency",
    "Nationality":                                 "nationality",
    "Citizenship":                                 "citizenship",
    "Residency":                                   "residency",
    "Date of Incorporation/Formation":             "incorporation_date",
    "Place of Incorporation/Formation":            "incorporation_place",
    "Beneficial Owner Types":                      "bo_types",
    "Passive NFE":                                 "passive_nfe",
    "Address of Registered Office":                "addr_registered",
    "Address of Place of Business":                "addr_business",
    "Address of Beneficial Owners/Related Persons":"addr_bo",
    "Address of Individual Customer":              "addr_individual",
    "Date of Birth":                               "dob",
    "Father/Spouse Name":                          "father_spouse",
    "Identification Proof Doc No":                 "id_doc_no",
    "Entity Identification Proof Doc No":          "entity_id_doc_no",
    "Credit Summation of the account for the period": "credit_sum_period",
    "Debit Summation of the account for the period":  "debit_sum_period",
    "Professional Experience in Years - Individual":   "experience_years",
    "CIF/ID of Beneficial Owners/Related Persons":     "cif_bo",
    "Name of Beneficial Owners/Related Persons":       "name_bo",
    "Mobile Number":                               "mobile",
    "PAN":                                         "pan",
    "Aadhaar Number":                              "aadhaar",
    "Email ID":                                    "email",
    "Wallet KYC Category":                         "wallet_kyc",
    "Wallet Account ID":                           "wallet_id",
    "Escrow Account Linked":                       "escrow_account",
    "Transaction Limit (Per Transaction)":         "limit_per_txn",
    "Daily Transaction Limit":                     "limit_daily",
    "Monthly Transaction Limit":                   "limit_monthly",
    "Annual Transaction Limit":                    "limit_annual",
    "Maximum Wallet Balance Limit":                "max_wallet_bal",
    "Device ID/Fingerprint":                       "device_id",
    "IP Address of Originating Device":            "ip_address",
    "Geo-Location (City/Country)":                 "geo_location",
    "GPS Coordinates":                             "gps_coords",
    "Browser/App Information":                     "browser_app",
    "Session ID":                                  "session_id",
    "Authentication Method (OTP/PIN/Biometric)":   "auth_method",
    "VPN Flag":                                    "vpn_flag",
    "Emulator Flag":                               "emulator_flag",
    "Lat/Long of Customer Address":                "customer_latlon",
}

def load_transactions(filepath):
    ext = filepath.rsplit('.', 1)[-1].lower()
    if ext == 'parquet':
        raw = pd.read_parquet(filepath)
    elif ext in ('xlsx', 'xls'):
        raw = pd.read_excel(filepath)
    else:
        raw = pd.read_csv(filepath, low_memory=False)
    print(f"Raw data loaded: {len(raw):,} rows x {len(raw.columns)} columns")

    # Handle duplicate "Transaction Type" columns
    cols = list(raw.columns)
    seen_txn_type = False
    for i, c in enumerate(cols):
        if c.strip() == "Transaction Type":
            if seen_txn_type:
                cols[i] = "Transaction Type PPI"
                COLUMN_MAP["Transaction Type PPI"] = "txn_type_ppi"
            seen_txn_type = True
    raw.columns = cols

    rename = {}
    for orig, clean in COLUMN_MAP.items():
        for col in raw.columns:
            if col.strip() == orig.strip():
                rename[col] = clean
                break
    raw = raw.rename(columns=rename)
    return raw

# Load
if os.path.exists(INPUT_FILE):
    df = load_transactions(INPUT_FILE)
else:
    print(f"'{INPUT_FILE}' not found. Looking for synthetic data...")
    for alt in ["../outputs_updated/transactions_generated_typology.parquet",
                "../outputs_updated/transactions_generated_typology.csv",
                "../transactions_generated_typology.parquet",
                "../transactions_generated_typology.csv"]:
        if os.path.exists(alt):
            df = load_transactions(alt)
            break
    else:
        raise FileNotFoundError("No transaction file found.")

Raw data loaded: 333,038 rows x 97 columns


## 4 -- Parse Datetime & Build Working Columns


In [34]:
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

COL = {
    "txn_id":    find_col(df, ["txn_id","transaction_id","Transaction ID/Reference No"]),
    "timestamp": find_col(df, ["timestamp","Timestamp"]),
    "datestamp":  find_col(df, ["datestamp","Datestamp"]),
    "amount":    find_col(df, ["amount","transaction_amount","Transaction Amount"]),
    "txn_type":  find_col(df, ["txn_type","transaction_type_dr_cr","Transaction Type"]),
    "channel":   find_col(df, ["channel_bank","transaction_mode_channel_bank","Transaction Mode/Channel - Bank"]),
    "cash_flag": find_col(df, ["cash_flag","Cash Flag"]),
    "status":    find_col(df, ["txn_status","transaction_status","Transaction Status"]),
    "acct":      find_col(df, ["account_number","customer_account_number","Customer Account Number"]),
    "cp_acct":   find_col(df, ["cp_account_number","counterparty_account_number","Counterparty Account Number"]),
    "cif":       find_col(df, ["cif_id","customer_cif_id","Customer CIF/ID Number"]),
}

print("Key columns resolved:")
for k, v in COL.items():
    print(f"  {k:<12s} -> {str(v):<45s} [{'OK' if v else 'MISSING'}]")

# Build working columns
def parse_dt(row):
    ds = str(row.get(COL["datestamp"], ""))
    ts = str(row.get(COL["timestamp"], "00:00:00"))
    for fmt in ["%d-%m-%Y %H:%M:%S","%d/%m/%Y %H:%M:%S","%Y-%m-%d %H:%M:%S","%d-%m-%Y","%Y-%m-%d"]:
        try:
            return datetime.strptime(f"{ds} {ts}".strip(), fmt)
        except:
            continue
    return pd.NaT

df["_dt"]   = df.apply(parse_dt, axis=1)
df["_amt"]  = pd.to_numeric(df[COL["amount"]], errors='coerce').fillna(0)
df["_acct"] = df[COL["acct"]].astype(str).str.strip()
df["_cp"]   = df[COL["cp_acct"]].astype(str).str.strip()
df["_type"] = df[COL["txn_type"]].astype(str).str.strip().str.upper() if COL["txn_type"] else "DR"
df["_cash"] = df[COL["cash_flag"]].astype(str).str.strip().str.upper() if COL["cash_flag"] else "N"
df["_stat"] = df[COL["status"]].astype(str).str.strip().str.upper() if COL["status"] else "SUCCESS"

# Only successful transactions for analysis
df_ok = df[df["_stat"].isin(["SUCCESS","COMPLETED","PROCESSED"])].copy()
df_ok = df_ok.sort_values("_dt").reset_index(drop=True)
# Store original df index for labeling back
df_ok["_orig_idx"] = df_ok.index  # after reset this is 0..N-1 in df_ok
# We need to map back to df's index; rebuild from df
df_ok = df[df["_stat"].isin(["SUCCESS","COMPLETED","PROCESSED"])].copy()
df_ok = df_ok.sort_values("_dt")

# Init label columns
df["is_aml"] = 0
df["aml_typology"] = ""
df["typology_group_id"] = ""

print(f"\nSuccessful transactions: {len(df_ok):,} / {len(df):,}")
print(f"Date range: {df_ok['_dt'].min()} to {df_ok['_dt'].max()}")



Key columns resolved:
  txn_id       -> transaction_id                                [OK]
  timestamp    -> timestamp                                     [OK]
  datestamp    -> datestamp                                     [OK]
  amount       -> transaction_amount                            [OK]
  txn_type     -> transaction_type_dr_cr                        [OK]
  channel      -> transaction_mode_channel_bank                 [OK]
  cash_flag    -> cash_flag                                     [OK]
  status       -> transaction_status                            [OK]
  acct         -> customer_account_number                       [OK]
  cp_acct      -> counterparty_account_number                   [OK]
  cif          -> customer_cif_id                               [OK]

Successful transactions: 309,449 / 333,038
Date range: 2025-12-01 03:26:08 to 2026-05-27 12:51:47


## 5 -- Build Transaction Indexes


In [35]:
print("Building indexes...")

# Per-account: all transactions sorted by time
acct_txns = defaultdict(list)
for orig_idx, row in df_ok.iterrows():
    acct_txns[row["_acct"]].append({
        "idx": orig_idx,  # index into the master df
        "dt": row["_dt"], "amt": row["_amt"],
        "type": row["_type"], "cash": row["_cash"],
        "cp": row["_cp"], "acct": row["_acct"],
    })

for k in acct_txns:
    acct_txns[k].sort(key=lambda x: x["dt"])

# Directed edges: determine sender/receiver from Dr/Cr
edges_out = defaultdict(list)  # sender -> [(receiver, amt, dt, idx), ...]
edges_in  = defaultdict(list)  # receiver -> [(sender, amt, dt, idx), ...]

for orig_idx, row in df_ok.iterrows():
    if row["_type"] in ("DR","DEBIT","D"):
        sender, receiver = row["_acct"], row["_cp"]
    else:
        sender, receiver = row["_cp"], row["_acct"]

    if sender and receiver and sender != receiver and sender != "nan" and receiver != "nan":
        e = {"sender":sender, "receiver":receiver, "amt":row["_amt"], "dt":row["_dt"], "idx":orig_idx}
        edges_out[sender].append(e)
        edges_in[receiver].append(e)

for k in edges_out: edges_out[k].sort(key=lambda x: x["dt"])
for k in edges_in:  edges_in[k].sort(key=lambda x: x["dt"])

print(f"  Unique accounts:  {len(acct_txns):,}")
print(f"  Directed edges:   {sum(len(v) for v in edges_out.values()):,}")

# Group-ID counter and result accumulator
_gcnt = defaultdict(int)
def next_gid(prefix):
    _gcnt[prefix] += 1
    return f"{prefix}_{_gcnt[prefix]:05d}"

results = []  # (df_index, typology, group_id)
def flag(indices, typology, gid):
    for i in indices:
        results.append((i, typology, gid))

print("Indexes ready")



Building indexes...
  Unique accounts:  4,763
  Directed edges:   306,158
Indexes ready


## 6 -- T1: Structuring (Smurfing)
**Pattern**: Multiple different accounts each make a sub-threshold cash deposit,
then all transfer funds to the **same target** account within a short window.
Detection is target-centric: find convergence of sub-threshold cash sources.


In [36]:
print("T1: Detecting Structuring (Smurfing)...")
cfg = DETECT_CONFIG["structuring"]
threshold   = cfg["cash_threshold"]
floor       = threshold * cfg["amount_floor_pct"]
win_days    = cfg["time_window_days"]
consol_days = cfg["consolidation_window_days"]
min_sources = cfg["min_source_accounts"]
t1_count = 0

# Step 1: For each account with sub-threshold cash deposits,
#         find subsequent outflows and record (source_acct, target_acct, deposit_idx, transfer_idx, dt)
links = []  # each: {src, tgt, dep_idx, xfer_idx, dep_dt}

for acct, txns in acct_txns.items():
    # Cash credits to this account that are sub-threshold
    cash_crs = [t for t in txns
                if t["cash"] == "Y"
                and t["type"] in ("CR","CREDIT","C")
                and floor <= t["amt"] < threshold]
    if not cash_crs:
        continue

    # Debits FROM this account (transfers out)
    debits = [t for t in txns
              if t["type"] in ("DR","DEBIT","D")
              and t["cp"] not in ("", "nan", acct)]

    for dep in cash_crs:
        deadline = dep["dt"] + timedelta(days=consol_days)
        for dr in debits:
            if dr["dt"] < dep["dt"]:
                continue
            if dr["dt"] > deadline:
                break
            links.append({
                "src": acct, "tgt": dr["cp"],
                "dep_idx": dep["idx"], "xfer_idx": dr["idx"],
                "dep_dt": dep["dt"],
            })
            break  # first matching debit per deposit

print(f"  Deposit-to-target links: {len(links):,}")

# Step 2: Group by target; find targets receiving from N+ distinct sources in a time window
tgt_links = defaultdict(list)
for lk in links:
    tgt_links[lk["tgt"]].append(lk)

for tgt, lks in tgt_links.items():
    if len(set(l["src"] for l in lks)) < min_sources:
        continue
    lks.sort(key=lambda x: x["dep_dt"])

    # Sliding window
    for i in range(len(lks)):
        w_end = lks[i]["dep_dt"] + timedelta(days=win_days)
        cluster = [lks[i]]
        for j in range(i+1, len(lks)):
            if lks[j]["dep_dt"] <= w_end:
                cluster.append(lks[j])
            else:
                break

        srcs = set(c["src"] for c in cluster)
        if len(srcs) >= min_sources:
            gid = next_gid("STRUCT")
            idxs = list(set(c["dep_idx"] for c in cluster) | set(c["xfer_idx"] for c in cluster))
            flag(idxs, "Structuring (Smurfing)", gid)
            t1_count += 1
            break  # one detection per target

print(f"  Structuring scenarios: {t1_count}")



T1: Detecting Structuring (Smurfing)...
  Deposit-to-target links: 3,588
  Structuring scenarios: 501


## 7 -- T2: Circular Transaction Loops
**Pattern**: A -> B -> C -> A within time window, similar amounts. DFS cycle detection.


In [37]:
print("T2: Detecting Circular Transaction Loops...")
cfg = DETECT_CONFIG["circular"]
win_days  = cfg["time_window_days"]
min_loop  = cfg["min_loop_size"]
max_loop  = cfg["max_loop_size"]
amt_tol   = cfg["amount_tolerance_pct"]
min_amt   = cfg["min_amount"]
max_other = cfg.get("max_other_txns_between_parties", 999)  # commercial filter
t2_count  = 0
seen_cycles = set()

# Collect high-value starting edges
starters = [e for elist in edges_out.values() for e in elist if e["amt"] >= min_amt]
starters.sort(key=lambda x: -x["amt"])
search_lim = min(len(starters), 20000)
print(f"  Searching from {search_lim:,} high-value edges...")

for si, start in enumerate(starters[:search_lim]):
    if si % 5000 == 0 and si > 0:
        print(f"    edge {si:,}/{search_lim:,}...")

    origin = start["sender"]
    stack = [(start, [start])]

    while stack:
        cur, path = stack.pop()
        if len(path) > max_loop:
            continue
        nxt_acct = cur["receiver"]
        deadline = start["dt"] + timedelta(days=win_days)

        if nxt_acct == origin and len(path) >= min_loop:
            cyc_key = tuple(sorted(e["idx"] for e in path))
            if cyc_key not in seen_cycles:
                # Commercial filter: check if cycle parties transact outside the cycle
                cycle_accts = list(set(e["sender"] for e in path))
                is_commercial = False
                if max_other < 999:
                    for ca in cycle_accts:
                        other_partners = set(e["receiver"] for e in edges_out.get(ca, [])
                                           if e["idx"] not in set(e2["idx"] for e2 in path))
                        cycle_partner_overlap = other_partners & set(cycle_accts)
                        if len(cycle_partner_overlap) > 0:
                            # Count non-cycle txns between these parties
                            other_count = sum(1 for e in edges_out.get(ca, [])
                                            if e["receiver"] in cycle_partner_overlap
                                            and e["idx"] not in set(e2["idx"] for e2 in path))
                            if other_count > max_other:
                                is_commercial = True
                                break

                if not is_commercial:
                    seen_cycles.add(cyc_key)
                    gid = next_gid("CIRC")
                    flag([e["idx"] for e in path], "Circular Transaction Loop", gid)
                    t2_count += 1
            continue

        for e in edges_out.get(nxt_acct, []):
            if e["dt"] < cur["dt"] or e["dt"] > deadline:
                if e["dt"] > deadline: break
                continue
            if abs(e["amt"] - start["amt"]) / max(start["amt"], 1) > amt_tol:
                continue
            visited = {ed["sender"] for ed in path}
            if e["receiver"] in visited and e["receiver"] != origin:
                continue
            stack.append((e, path + [e]))

print(f"  Circular loop scenarios: {t2_count}")



T2: Detecting Circular Transaction Loops...
  Searching from 20,000 high-value edges...
    edge 5,000/20,000...
    edge 10,000/20,000...
    edge 15,000/20,000...
  Circular loop scenarios: 2957


## 8 -- T3: Funnel Account Networks
**Pattern**: Many unrelated senders -> one account -> rapid large outflow.


In [38]:
print("T3: Detecting Funnel Account Networks...")
cfg = DETECT_CONFIG["funnel"]
win_days    = cfg["time_window_days"]
min_senders = cfg["min_unique_senders"]
out_days    = cfg["outflow_window_days"]
retention   = cfg["outflow_retention_pct"]
min_inflow  = cfg["min_total_inflow"]
t3_count = 0

for recv, inflows in edges_in.items():
    if len(inflows) < min_senders:
        continue

    for i in range(len(inflows)):
        w_end = inflows[i]["dt"] + timedelta(days=win_days)
        win_in = [inflows[i]]
        for j in range(i+1, len(inflows)):
            if inflows[j]["dt"] <= w_end:
                win_in.append(inflows[j])
            else:
                break

        if len(set(e["sender"] for e in win_in)) < min_senders:
            continue

        total_in = sum(e["amt"] for e in win_in)
        if total_in < min_inflow:
            continue

        last_in = max(e["dt"] for e in win_in)
        out_deadline = last_in + timedelta(days=out_days)
        outs = [e for e in edges_out.get(recv, []) if last_in <= e["dt"] <= out_deadline]
        total_out = sum(e["amt"] for e in outs)

        if total_out >= total_in * (1 - retention):
            gid = next_gid("FUNNEL")
            idxs = [e["idx"] for e in win_in] + [e["idx"] for e in outs]
            flag(idxs, "Funnel Account Network", gid)
            t3_count += 1
            break

print(f"  Funnel network scenarios: {t3_count}")



T3: Detecting Funnel Account Networks...
  Funnel network scenarios: 2186


## 9 -- T4: Pass-Through Transit Hubs
**Pattern**: Receive large sum, forward 90%+ within minutes, retain almost nothing.


In [39]:
print("T4: Detecting Pass-Through Transit Hubs...")
cfg = DETECT_CONFIG["passthrough"]
gap_min   = cfg["time_gap_minutes"]
ret_pct   = cfg["retention_pct"]
min_amt   = cfg["min_amount"]
min_occ   = cfg["min_occurrences"]
require_diff_cp = cfg.get("require_different_counterparties", False)
max_net_ratio   = cfg.get("max_net_position_ratio", 1.0)
t4_count  = 0

# Pre-compute net position ratio per account (total_credits / total_debits)
acct_net_ratio = {}
if max_net_ratio < 1.0:
    for acct, txns in acct_txns.items():
        total_cr = sum(t["amt"] for t in txns if t["type"] in ("CR","CREDIT","C"))
        total_dr = sum(t["amt"] for t in txns if t["type"] in ("DR","DEBIT","D"))
        if total_cr + total_dr > 0:
            acct_net_ratio[acct] = abs(total_cr - total_dr) / (total_cr + total_dr)
        else:
            acct_net_ratio[acct] = 0

acct_events = defaultdict(list)

for acct in set(edges_in.keys()) & set(edges_out.keys()):
    # Net position filter: skip accounts that clearly retain funds
    if max_net_ratio < 1.0:
        if acct_net_ratio.get(acct, 1.0) > max_net_ratio:
            continue

    big_in = [e for e in edges_in[acct] if e["amt"] >= min_amt]
    outs   = edges_out[acct]

    for inf in big_in:
        deadline = inf["dt"] + timedelta(minutes=gap_min)
        matched = [o for o in outs
                   if inf["dt"] <= o["dt"] <= deadline
                   and o["receiver"] != inf["sender"]]

        # Counterparty diversity: inflow source must differ from outflow destinations
        if require_diff_cp and matched:
            matched = [o for o in matched if o["receiver"] != inf["sender"]]

        if not matched:
            continue
        total_out = sum(o["amt"] for o in matched)
        kept = inf["amt"] - total_out
        if 0 <= kept <= inf["amt"] * ret_pct:
            acct_events[acct].append({"inflow": inf, "outflows": matched})

for acct, evts in acct_events.items():
    if len(evts) >= min_occ:
        for evt in evts:
            gid = next_gid("PASS")
            idxs = [evt["inflow"]["idx"]] + [o["idx"] for o in evt["outflows"]]
            flag(idxs, "Pass-Through Transit Hub", gid)
            t4_count += 1

print(f"  Pass-through scenarios: {t4_count}")



T4: Detecting Pass-Through Transit Hubs...
  Pass-through scenarios: 1578


## 10 -- T5: Rapid Multi-Hop Layering
**Pattern**: Funds traverse 4+ accounts within hours. BFS forward chain tracing.


In [40]:
print("T5: Detecting Rapid Multi-Hop Layering...")
cfg = DETECT_CONFIG["layering"]
max_hrs   = cfg["max_chain_hours"]
min_hops  = cfg["min_hops"]
max_hops  = cfg["max_hops"]
decay_tol = cfg["amount_decay_tolerance"]
min_amt   = cfg["min_amount"]
search_limit_cfg = cfg.get("search_limit", 15000)
t5_count  = 0
used_edges = set()

starters = [e for elist in edges_out.values() for e in elist if e["amt"] >= min_amt]
starters.sort(key=lambda x: -x["amt"])
search_lim = min(len(starters), search_limit_cfg)
print(f"  Tracing from {search_lim:,} high-value edges (search_limit={search_limit_cfg:,})...")

for si, start in enumerate(starters[:search_lim]):
    if si % 5000 == 0 and si > 0:
        print(f"    edge {si:,}/{search_lim:,}...")
    if start["idx"] in used_edges:
        continue

    deadline = start["dt"] + timedelta(hours=max_hrs)
    best = [start]
    queue = deque([(start, [start])])

    while queue:
        cur, chain = queue.popleft()
        if len(chain) > max_hops:
            continue
        nxt = cur["receiver"]
        for e in edges_out.get(nxt, []):
            if e["dt"] < cur["dt"] or e["dt"] > deadline:
                if e["dt"] > deadline: break
                continue
            if e["amt"] < start["amt"] * (1 - decay_tol) or e["amt"] > start["amt"] * 1.05:
                continue
            visited = {ed["sender"] for ed in chain} | {chain[-1]["receiver"]}
            if e["receiver"] in visited:
                continue
            new_chain = chain + [e]
            if len(new_chain) > len(best):
                best = new_chain
            queue.append((e, new_chain))

    if len(best) >= min_hops:
        gid = next_gid("LAYER")
        idxs = [e["idx"] for e in best]
        for i in idxs:
            used_edges.add(i)
        flag(idxs, "Rapid Multi-Hop Layering", gid)
        t5_count += 1

print(f"  Multi-hop layering scenarios: {t5_count}")



T5: Detecting Rapid Multi-Hop Layering...
  Tracing from 50,000 high-value edges (search_limit=50,000)...
    edge 5,000/50,000...
    edge 10,000/50,000...
    edge 15,000/50,000...
    edge 20,000/50,000...
    edge 25,000/50,000...
    edge 30,000/50,000...
    edge 35,000/50,000...
    edge 40,000/50,000...
    edge 45,000/50,000...
  Multi-hop layering scenarios: 755


## 11 -- Apply Labels


## 11 -- T6: Third-Party Payment Webs
**Pattern**: Business receives payments from many unrelated individuals not matching its customer base.


In [42]:
print("T6: Detecting Third-Party Payment Webs...")
cfg = DETECT_CONFIG["third_party_web"]
win_days = cfg["time_window_days"]
min_payers = cfg["min_unique_payers"]
pay_lo, pay_hi = cfg["per_payment_amount_range"]
valid_ch = set(cfg["channels"])
hr_lo, hr_hi = cfg["hour_range"]
t6_count = 0

for recv, inflows in edges_in.items():
    valid_in = []
    for e in inflows:
        if not (pay_lo <= e["amt"] <= pay_hi):
            continue
        # Channel filter (safe access — skip if not available)
        if "channel" in e and e["channel"] not in valid_ch:
            continue
        # Hour filter (safe access — skip if not available)
        if "hour" in e and not (hr_lo <= e["hour"] <= hr_hi):
            continue
        valid_in.append(e)

    if len(valid_in) < min_payers:
        continue

    for i in range(len(valid_in)):
        w_end = valid_in[i]["dt"] + timedelta(days=win_days)
        cluster = [valid_in[i]]
        for j in range(i + 1, len(valid_in)):
            if valid_in[j]["dt"] <= w_end:
                cluster.append(valid_in[j])
            else:
                break
        senders = set(e["sender"] for e in cluster)
        if len(senders) >= min_payers:
            gid = next_gid("TPWEB")
            flag([e["idx"] for e in cluster], "Third-Party Payment Web", gid)
            t6_count += 1
            break

print(f"  Third-party payment web scenarios: {t6_count}")

T6: Detecting Third-Party Payment Webs...
  Third-party payment web scenarios: 4724


## 12 -- T7: Money Mule Networks
**Pattern**: Central controller sends to many mules, each mule forwards to a collector.


In [44]:
print("T7: Detecting Money Mule Networks...")
cfg = DETECT_CONFIG["money_mule"]
min_mules = cfg["min_mules"]
amt_lo, amt_hi = cfg["controller_amount_range"]
fwd_lo, fwd_hi = cfg["forward_pct_range"]
fwd_delay_hrs = cfg["forward_delay_hours"]
valid_ch = set(cfg["channels"])
hr_lo, hr_hi = cfg["hour_range"]
t7_count = 0

# Find controllers: accounts sending to many distinct receivers with similar amounts
for controller, outflows in edges_out.items():
    valid_out = []
    for e in outflows:
        if not (amt_lo <= e["amt"] <= amt_hi):
            continue
        if "channel" in e and e["channel"] not in valid_ch:
            continue
        if "hour" in e and not (hr_lo <= e["hour"] <= hr_hi):
            continue
        valid_out.append(e)

    receivers = set(e["receiver"] for e in valid_out)
    if len(receivers) < min_mules:
        continue

    # Check if receivers (mules) forward to a common collector
    mule_fwd_map = defaultdict(list)
    for mule_acct in receivers:
        mule_outs = []
        for e in edges_out.get(mule_acct, []):
            if "channel" in e and e["channel"] not in valid_ch:
                continue
            mule_outs.append(e)

        for mo in mule_outs:
            mule_ins = [vi for vi in valid_out if vi["receiver"] == mule_acct]
            for mi in mule_ins:
                gap_hrs = (mo["dt"] - mi["dt"]).total_seconds() / 3600
                if 0 <= gap_hrs <= fwd_delay_hrs:
                    if fwd_lo * mi["amt"] <= mo["amt"] <= fwd_hi * mi["amt"]:
                        mule_fwd_map[mo["receiver"]].append({"in": mi, "out": mo})

    # Find collector with most mule connections
    for collector, fwd_events in mule_fwd_map.items():
        if len(fwd_events) >= min_mules:
            gid = next_gid("MULE")
            idxs = []
            for fe in fwd_events:
                idxs.extend([fe["in"]["idx"], fe["out"]["idx"]])
            flag(list(set(idxs)), "Money Mule Network", gid)
            t7_count += 1
            break

print(f"  Money mule network scenarios: {t7_count}")

T7: Detecting Money Mule Networks...
  Money mule network scenarios: 114


## 13 -- T8: High-Risk Corridor Transfers
**Pattern**: Repeated transfers from one account to high-risk FATF countries.


In [46]:
print("T8: Detecting High-Risk Corridor Transfers...")
cfg = DETECT_CONFIG["high_risk_corridor"]
amt_lo, amt_hi = cfg["amount_range"]
target_countries = set(cfg["target_countries"])
min_xfers = cfg["min_transfers_per_account"]
win_days = cfg["time_window_days"]
valid_ch = set(cfg["channels"])
hr_lo, hr_hi = cfg["hour_range"]
t8_count = 0

# Need receiver country code column
rc_col = find_col(df, ["receiver_country_code", "receiver_country", "Receiver Country Code*"])
if rc_col:
    for acct, txns in acct_txns.items():
        corridor_txns = []
        for t in txns:
            idx = t["idx"]
            if idx not in df.index:
                continue
            recv_cc = str(df.at[idx, rc_col]).strip().upper() if rc_col in df.columns else ""
            if recv_cc not in target_countries:
                continue
            if not (amt_lo <= t["amt"] <= amt_hi):
                continue
            if "channel" in t and t["channel"] not in valid_ch:
                continue
            if "hour" in t and not (hr_lo <= t["hour"] <= hr_hi):
                continue
            if t["type"] not in ("DR", "DEBIT", "D"):
                continue
            corridor_txns.append(t)

        if len(corridor_txns) < min_xfers:
            continue

        corridor_txns.sort(key=lambda x: x["dt"])
        for i in range(len(corridor_txns)):
            w_end = corridor_txns[i]["dt"] + timedelta(days=win_days)
            cluster = [corridor_txns[i]]
            for j in range(i + 1, len(corridor_txns)):
                if corridor_txns[j]["dt"] <= w_end:
                    cluster.append(corridor_txns[j])
                else:
                    break
            if len(cluster) >= min_xfers:
                gid = next_gid("HRCORR")
                flag([t["idx"] for t in cluster], "High-Risk Corridor Transfer", gid)
                t8_count += 1
                break
else:
    print("  WARNING: No receiver country column found, skipping")

print(f"  High-risk corridor scenarios: {t8_count}")

T8: Detecting High-Risk Corridor Transfers...
  High-risk corridor scenarios: 517


## 14 -- T9: Underground Banking (Hawala)
**Pattern**: Triangular/quadrilateral settlements where amounts match across 3-4 parties.


In [47]:
print("T9: Detecting Underground Banking (Hawala)...")
cfg = DETECT_CONFIG["hawala"]
min_parties, max_parties = cfg["num_parties_range"]
amt_lo, amt_hi = cfg["amount_range"]
amt_tol = cfg["amount_tolerance_pct"]
win_days = cfg["time_window_days"]
valid_ch = set(cfg["channels"])
hr_lo, hr_hi = cfg["hour_range"]
t9_count = 0
seen_hawala = set()

# Filter starters with safe access
starters = []
for elist in edges_out.values():
    for e in elist:
        if not (amt_lo <= e["amt"] <= amt_hi):
            continue
        if "channel" in e and e["channel"] not in valid_ch:
            continue
        if "hour" in e and not (hr_lo <= e["hour"] <= hr_hi):
            continue
        starters.append(e)

starters.sort(key=lambda x: -x["amt"])
search_lim = min(len(starters), 15000)

for si, start in enumerate(starters[:search_lim]):
    origin = start["sender"]
    stack = [(start, [start])]
    while stack:
        cur, path = stack.pop()
        if len(path) > max_parties:
            continue
        nxt = cur["receiver"]
        deadline = start["dt"] + timedelta(days=win_days)
        if nxt == origin and min_parties <= len(path) <= max_parties:
            cyc_key = tuple(sorted(e["idx"] for e in path))
            if cyc_key not in seen_hawala:
                seen_hawala.add(cyc_key)
                gid = next_gid("HAWALA")
                flag([e["idx"] for e in path], "Underground Banking (Hawala)", gid)
                t9_count += 1
            continue
        for e in edges_out.get(nxt, []):
            if e["dt"] < cur["dt"] or e["dt"] > deadline:
                if e["dt"] > deadline: break
                continue
            if "channel" in e and e["channel"] not in valid_ch:
                continue
            if "hour" in e and not (hr_lo <= e["hour"] <= hr_hi):
                continue
            if abs(e["amt"] - start["amt"]) / max(start["amt"], 1) > amt_tol:
                continue
            visited = {ed["sender"] for ed in path}
            if e["receiver"] in visited and e["receiver"] != origin:
                continue
            stack.append((e, path + [e]))

print(f"  Hawala scenarios: {t9_count}")

T9: Detecting Underground Banking (Hawala)...
  Hawala scenarios: 2612


## 15 -- T10: Charity Abuse
**Pattern**: NPO collects many donations then diverts funds to personal accounts.


In [48]:
print("T10: Detecting Charity Abuse...")
cfg = DETECT_CONFIG["charity_abuse"]
min_donors = cfg["min_donors"]
don_lo, don_hi = cfg["donation_amount_range"]
don_win = cfg["donation_window_days"]
div_win = cfg["diversion_window_days"]
div_ret = cfg["diversion_retention_pct"]
don_channels = set(cfg["donation_channels"])
div_hr_lo, div_hr_hi = cfg["diversion_hour_range"]
t10_count = 0

for recv, inflows in edges_in.items():
    valid_donations = []
    for e in inflows:
        if not (don_lo <= e["amt"] <= don_hi):
            continue
        if "channel" in e and e["channel"] not in don_channels:
            continue
        valid_donations.append(e)

    if len(valid_donations) < min_donors:
        continue

    for i in range(len(valid_donations)):
        w_end = valid_donations[i]["dt"] + timedelta(days=don_win)
        cluster = [valid_donations[i]]
        for j in range(i + 1, len(valid_donations)):
            if valid_donations[j]["dt"] <= w_end:
                cluster.append(valid_donations[j])
            else:
                break

        donors = set(e["sender"] for e in cluster)
        if len(donors) < min_donors:
            continue

        total_donated = sum(e["amt"] for e in cluster)
        last_donation = max(e["dt"] for e in cluster)
        div_deadline = last_donation + timedelta(days=div_win)

        diversions = []
        for e in edges_out.get(recv, []):
            if not (last_donation <= e["dt"] <= div_deadline):
                continue
            if "hour" in e and not (div_hr_lo <= e["hour"] <= div_hr_hi):
                continue
            diversions.append(e)

        total_diverted = sum(e["amt"] for e in diversions)
        if total_diverted >= total_donated * (1 - div_ret):
            gid = next_gid("CHARITY")
            idxs = [e["idx"] for e in cluster] + [e["idx"] for e in diversions]
            flag(idxs, "Charity Abuse", gid)
            t10_count += 1
            break

print(f"  Charity abuse scenarios: {t10_count}")

T10: Detecting Charity Abuse...
  Charity abuse scenarios: 4787


In [49]:
print("Applying labels to master dataframe...")

label_map = defaultdict(lambda: {"typ": [], "grp": []})
for idx, typ, gid in results:
    label_map[idx]["typ"].append(typ)
    label_map[idx]["grp"].append(gid)

flagged_count = 0
for idx, lab in label_map.items():
    if idx in df.index:
        df.at[idx, "is_aml"] = 1
        df.at[idx, "aml_typology"] = "; ".join(sorted(set(lab["typ"])))
        df.at[idx, "typology_group_id"] = "; ".join(sorted(set(lab["grp"])))
        flagged_count += 1

total = len(df)
print(f"\n{'='*70}")
print(f"DETECTION SUMMARY")
print(f"{'='*70}")
print(f"  Total transactions:  {total:>10,}")
print(f"  Flagged as AML:      {flagged_count:>10,}  ({flagged_count/total*100:.2f}%)")
print(f"  Clean:               {total-flagged_count:>10,}  ({(total-flagged_count)/total*100:.2f}%)")
print(f"{'='*70}")



Applying labels to master dataframe...

DETECTION SUMMARY
  Total transactions:     333,038
  Flagged as AML:         130,136  (39.08%)
  Clean:                  202,902  (60.92%)


## 12 -- Typology Distribution & Scenario Summary


In [50]:
flagged = df[df["is_aml"] == 1]

if len(flagged) > 0:
    all_typ = []
    grp_sets = defaultdict(set)
    for _, row in flagged.iterrows():
        typs = str(row["aml_typology"]).split("; ")
        grps = str(row["typology_group_id"]).split("; ")
        for t in typs:
            all_typ.append(t)
        for t, g in zip(typs, grps):
            grp_sets[t].add(g)

    tc = Counter(all_typ)
    total_f = len(flagged)

    print(f"\n{'Typology':<37s} {'Txns':>8s} {'Scenarios':>10s} {'% Flagged':>10s}")
    print("-" * 70)
    for t in sorted(tc):
        print(f"  {t:<35s} {tc[t]:>8,} {len(grp_sets[t]):>10,} {tc[t]/total_f*100:>9.1f}%")
    print("-" * 70)
    print(f"  {'TOTAL FLAGGED':<35s} {total_f:>8,}")
    print(f"  {'TOTAL SCENARIOS':<35s} {sum(len(v) for v in grp_sets.values()):>8,}")

    # Sample
    scols = [c for c in [COL["txn_id"], COL["datestamp"], COL["amount"],
                          COL["acct"], COL["cp_acct"], "aml_typology", "typology_group_id"]
             if c and c in flagged.columns]
    print(f"\nSample flagged transactions:")
    print(flagged[scols].head(10).to_string(index=False))
else:
    print("No AML transactions detected. Consider loosening thresholds in DETECT_CONFIG.")




Typology                                  Txns  Scenarios  % Flagged
----------------------------------------------------------------------
  Charity Abuse                         85,723      4,787      65.9%
  Circular Transaction Loop              1,319        823       1.0%
  Funnel Account Network                46,993      3,404      36.1%
  High-Risk Corridor Transfer            2,992        526       2.3%
  Money Mule Network                     1,418        136       1.1%
  Pass-Through Transit Hub               3,087      1,425       2.4%
  Rapid Multi-Hop Layering               2,542      1,243       2.0%
  Structuring (Smurfing)                 2,913        565       2.2%
  Third-Party Payment Web               22,032      6,042      16.9%
  Underground Banking (Hawala)           1,671      1,043       1.3%
----------------------------------------------------------------------
  TOTAL FLAGGED                        130,136
  TOTAL SCENARIOS                       19,994

Sam

## 13 -- Export Labeled Datasets


In [51]:
# Drop internal working columns
internal = [c for c in df.columns if c.startswith("_")]
df_out = df.drop(columns=internal)

# Full labeled dataset (parquet)
path_full = os.path.join(OUTPUT_DIR, "stg_transactions_typology.parquet")
df_out.to_parquet(path_full, index=False, compression='snappy')
print(f"Full labeled: {path_full}")
print(f"  {len(df_out):,} rows x {len(df_out.columns)} columns")
print(f"  Size: {os.path.getsize(path_full) / (1024*1024):.2f} MB")

# Flagged-only (parquet)
df_flagged = df_out[df_out["is_aml"] == 1]
path_flag = os.path.join(OUTPUT_DIR, "flagged_transactions.parquet")
df_flagged.to_parquet(path_flag, index=False, compression='snappy')
print(f"\nFlagged only: {path_flag}")
print(f"  {len(df_flagged):,} rows")

# Scenario summary
if len(df_flagged) > 0:
    rows = []
    amt_col = COL["amount"] or "amount"
    acct_col = COL["acct"] or "account_number"
    for _, r in df_flagged.iterrows():
        for g in str(r["typology_group_id"]).split("; "):
            rows.append({"group_id": g, "typology": r["aml_typology"],
                         "amount": r.get(amt_col, 0), "account": r.get(acct_col, "")})
    sdf = pd.DataFrame(rows)
    sdf["amount"] = pd.to_numeric(sdf["amount"], errors="coerce")
    agg = sdf.groupby(["group_id","typology"]).agg(
        txn_count=("amount","count"), total_amount=("amount","sum"),
        unique_accounts=("account","nunique")
    ).reset_index().sort_values("total_amount", ascending=False)
    path_sc = os.path.join(OUTPUT_DIR, "scenario_summary.csv")
    agg.to_csv(path_sc, index=False)
    print(f"\nScenario summary: {path_sc}")
    print(f"  {len(agg):,} scenarios")
    print(agg.head(15).to_string(index=False))

print(f"\n{'='*60}")
print(f"DETECTION PIPELINE COMPLETE")
print(f"All outputs: {os.path.abspath(OUTPUT_DIR)}/")
print(f"{'='*60}")

Full labeled: ../outputs_updated\stg_transactions_typology.parquet
  333,038 rows x 97 columns
  Size: 39.69 MB

Flagged only: ../outputs_updated\flagged_transactions.parquet
  130,136 rows

Scenario summary: ../outputs_updated\scenario_summary.csv
  45,659 scenarios
     group_id                              typology  txn_count  total_amount  unique_accounts
 FUNNEL_00675 Charity Abuse; Funnel Account Network         12   10983816.30                7
CHARITY_01314 Charity Abuse; Funnel Account Network         11   10879826.27                6
CHARITY_00423                         Charity Abuse         14   10511371.44                7
 FUNNEL_00405                Funnel Account Network         20   10382722.02                8
CHARITY_00616                         Charity Abuse         10    9943027.00                3
 FUNNEL_00558                Funnel Account Network         33    8940489.28               14
CHARITY_03251                         Charity Abuse         18    7926883.